# Notebook 03: YOLOv8 Training & Evaluation

This notebook trains a **YOLOv8** object detection model for PCB defect detection using transfer learning from COCO-pretrained weights.

**Designed for Google Colab with GPU runtime** (T4 16GB VRAM recommended).

**Pipeline:**
1. Load pretrained YOLOv8n (nano) — 3.2M parameters
2. Fine-tune on the Kaggle PCB Defects dataset (6 classes)
3. Evaluate with mAP, precision, recall, per-class metrics
4. Visualize training curves, predictions, and confusion matrix
5. Save best model for downstream notebooks

In [ ]:
# Colab setup (uncomment when running on Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install -q ultralytics

import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO

# Paths — update for Colab if needed
DATASET_YAML = Path("../data/pcb-yolo/dataset.yaml")
PROJECT_DIR = Path("../models")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    0: "missing_hole", 1: "mouse_bite", 2: "open_circuit",
    3: "short", 4: "spur", 5: "spurious_copper",
}

print(f"Dataset config: {DATASET_YAML}")

## 1. Load Pretrained YOLOv8

We start with **YOLOv8n** (nano, 3.2M params) pretrained on COCO for fast iteration. If mAP@0.5 < 0.70, we can upgrade to YOLOv8s (11.2M params).

In [ ]:
# Load YOLOv8n pretrained on COCO
model = YOLO("yolov8n.pt")
print(f"Model loaded: YOLOv8n")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

## 2. Training Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `epochs` | 100 | Sufficient for convergence with early stopping |
| `patience` | 10 | Stop if no improvement for 10 epochs |
| `imgsz` | 640 | YOLOv8 default — good balance of detail vs memory |
| `batch` | 16 | Sized for Colab T4 (16GB VRAM) — reduce to 8 if OOM |
| `optimizer` | AdamW | Better generalization than SGD for fine-tuning |
| `lr0` | 0.01 | Default Ultralytics learning rate |
| `flipud` | 0.5 | PCBs have no "up" — enable vertical flip |
| `degrees` | 90 | PCBs can be rotated — enable rotation |
| `mosaic` | 1.0 | Helps with small defect detection |

In [ ]:
# Train YOLOv8 on PCB defect dataset
results = model.train(
    data=str(DATASET_YAML),
    epochs=100,
    patience=10,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.01,
    # Augmentation tuned for PCB images
    flipud=0.5,
    degrees=90.0,
    mosaic=1.0,
    # Checkpointing
    project=str(PROJECT_DIR),
    name="yolov8_pcb",
    save=True,
    save_period=10,
    # Uncomment to resume interrupted training:
    # resume=True,
)

## 3. Training Curves

We plot loss curves and validation metrics across epochs to verify convergence and check for overfitting.

In [ ]:
# Plot training curves from results CSV
results_csv = PROJECT_DIR / "yolov8_pcb" / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    metrics = [
        ("train/box_loss", "Box Loss (Train)"),
        ("train/cls_loss", "Classification Loss (Train)"),
        ("train/dfl_loss", "DFL Loss (Train)"),
        ("metrics/precision(B)", "Precision"),
        ("metrics/recall(B)", "Recall"),
        ("metrics/mAP50(B)", "mAP@0.5"),
    ]

    for ax, (col, title) in zip(axes.flat, metrics):
        if col in df.columns:
            ax.plot(df["epoch"], df[col], linewidth=2)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(col.split("/")[-1])
            ax.set_title(title)
            ax.grid(True, alpha=0.3)

    plt.suptitle("YOLOv8 Training Curves — PCB Defect Detection", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"Results not found at {results_csv} — complete training first")

## 4. Validation Evaluation

Run the best model on the held-out test split and report mAP, precision, recall, and per-class AP.

In [ ]:
# Evaluate best model on test set
best_model_path = PROJECT_DIR / "yolov8_pcb" / "weights" / "best.pt"

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    val_results = best_model.val(data=str(DATASET_YAML), split="test")

    print("=== Test Set Evaluation ===")
    print(f"mAP@0.5:      {val_results.box.map50:.4f}")
    print(f"mAP@0.5:0.95: {val_results.box.map:.4f}")
    print(f"Precision:     {val_results.box.mp:.4f}")
    print(f"Recall:        {val_results.box.mr:.4f}")

    print("\nPer-class AP@0.5:")
    for i, ap in enumerate(val_results.box.ap50):
        print(f"  {CLASS_NAMES[i]}: {ap:.4f}")
else:
    print(f"Best model not found at {best_model_path}")

## 5. Prediction Visualization

Visualize YOLOv8 predictions on test images — both correct detections and failure cases.

In [ ]:
# Visualize predictions on test images
test_images_dir = Path("../data/pcb-yolo/images/test")
test_images = sorted(test_images_dir.glob("*"))[:6]

if best_model_path.exists() and test_images:
    best_model = YOLO(str(best_model_path))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, img_path in zip(axes.flat, test_images):
        results = best_model.predict(str(img_path), verbose=False)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        ax.imshow(annotated_rgb)
        n_det = len(results[0].boxes)
        ax.set_title(f"{img_path.stem} ({n_det} detections)")
        ax.axis("off")

    plt.suptitle("YOLOv8 Predictions on Test Images", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Model or test images not available")

## 6. Confusion Matrix

In [ ]:
# Display confusion matrices from validation
cm_path = PROJECT_DIR / "yolov8_pcb" / "confusion_matrix.png"
cm_norm_path = PROJECT_DIR / "yolov8_pcb" / "confusion_matrix_normalized.png"

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, path, title in [
    (axes[0], cm_path, "Confusion Matrix"),
    (axes[1], cm_norm_path, "Normalized Confusion Matrix"),
]:
    if path.exists():
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(title)
    else:
        ax.text(0.5, 0.5, f"Not found:\n{path.name}", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Save Best Model

Copy the best weights to the project `models/` directory for use in downstream notebooks (04-ResNet, 06-Comparison, 07-ONNX, 08-Flask).

In [ ]:
# Save best model for downstream use
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

if best_model_path.exists():
    dst = models_dir / "yolov8_best.pt"
    shutil.copy2(best_model_path, dst)
    print(f"Best model saved to {dst}")
    print(f"Model size: {dst.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print("No best model to save — complete training first")